In [ ]:
import cv2
import time
import math
import numpy as np
from collections import deque
import mediapipe as mp
import requests

# ====== StreamReader for ESP ======
def open_with_opencv(url, prefer_ffmpeg=True, timeout=5.0):
    backends = []
    try:
        if prefer_ffmpeg and hasattr(cv2, 'CAP_FFMPEG'):
            backends.append(cv2.CAP_FFMPEG)
    except Exception:
        pass
    backends.append(cv2.CAP_ANY)

    for b in backends:
        try:
            cap = cv2.VideoCapture(url, b)
        except Exception:
            try:
                cap = cv2.VideoCapture(url)
            except Exception:
                cap = None
        if cap is None:
            continue
        t0 = time.time()
        while time.time() - t0 < timeout:
            if cap.isOpened():
                return cap
            time.sleep(0.1)
        try:
            cap.release()
        except Exception:
            pass
    return None


def mjpeg_stream_generator(url, timeout=10):
    sess = requests.Session()
    resp = sess.get(url, stream=True, timeout=timeout)
    if resp.status_code != 200:
        resp.close()
        raise IOError(f"HTTP {resp.status_code}")
    bytes_buf = b''
    for chunk in resp.iter_content(chunk_size=1024):
        if not chunk:
            continue
        bytes_buf += chunk
        start = bytes_buf.find(b'\xff\xd8')
        end = bytes_buf.find(b'\xff\xd9')
        if start != -1 and end != -1 and end > start:
            jpg = bytes_buf[start:end+2]
            bytes_buf = bytes_buf[end+2:]
            img = cv2.imdecode(np.frombuffer(jpg, dtype=np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                continue
            yield img
    resp.close()


class StreamReader:
    def __init__(self, url):
        self.url = url
        self.cap = None
        self.mjpeg_gen = None

    def connect(self):
        self.close()
        self.cap = open_with_opencv(self.url)
        if self.cap:
            print("✓ Opened stream via OpenCV VideoCapture")
            return True
        try:
            self.mjpeg_gen = mjpeg_stream_generator(self.url)
            print("✓ Using MJPEG fallback (requests)")
            return True
        except Exception as e:
            print(f"✗ Failed to open stream: {e}")
            self.mjpeg_gen = None
            return False

    def read(self):
        if self.cap:
            try:
                ok, frame = self.cap.read()
            except Exception:
                ok, frame = False, None
            if ok and frame is not None:
                return frame
            try:
                self.cap.release()
            except Exception:
                pass
            self.cap = None
        if self.mjpeg_gen:
            try:
                frame = next(self.mjpeg_gen)
                return frame
            except Exception:
                self.mjpeg_gen = None
        return None

    def close(self):
        if self.cap:
            try:
                self.cap.release()
            except Exception:
                pass
            self.cap = None
        self.mjpeg_gen = None


# ====== Utility Functions ======
def euclidean_distance(p1, p2):
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

def calculate_ear(landmarks, eye_indices):
    top = landmarks[eye_indices[0]]
    bottom = landmarks[eye_indices[1]]
    left = landmarks[eye_indices[2]]
    right = landmarks[eye_indices[3]]
    v1 = landmarks[eye_indices[4]]
    v2 = landmarks[eye_indices[5]]

    vertical_1 = euclidean_distance(top, bottom)
    vertical_2 = euclidean_distance(v1, v2)
    horizontal = euclidean_distance(left, right)
    return (vertical_1 + vertical_2) / (2.0 * horizontal) if horizontal != 0 else 0.0

def calculate_mar(landmarks):
    upper = landmarks[13]
    lower = landmarks[14]
    left = landmarks[78]
    right = landmarks[308]
    vertical = euclidean_distance(upper, lower)
    horizontal = euclidean_distance(left, right)
    return vertical / horizontal if horizontal != 0 else 0.0

IDX_NOSE_TIP = 1
IDX_CHIN = 152
IDX_LEFT_EYE_OUTER = 33
IDX_RIGHT_EYE_OUTER = 263
IDX_LEFT_MOUTH = 61
IDX_RIGHT_MOUTH = 291

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),
    (0.0, -63.6, -12.5),
    (-43.3, 32.7, -26.0),
    (43.3, 32.7, -26.0),
    (-28.9, -28.9, -24.1),
    (28.9, -28.9, -24.1)
], dtype=np.float32)

def wrap180(a):
    return float((a + 180.0) % 360.0 - 180.0)

def rotationMatrixToEulerAngles(R):
    sy = math.sqrt(R[0,0]**2 + R[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.atan2(R[2,1], R[2,2])
        yaw = math.atan2(-R[2,0], sy)
        roll = math.atan2(R[1,0], R[0,0])
    else:
        pitch = math.atan2(-R[1,2], R[1,1])
        yaw = math.atan2(-R[2,0], sy)
        roll = 0.0
    pitch, yaw, roll = np.degrees([pitch, yaw, roll])
    return wrap180(pitch), wrap180(yaw), wrap180(roll)

# ====== CONFIG ======
SMOOTH_N = 15
PITCH_DELTA_ON = 10.0
PITCH_DELTA_OFF = 7.0
PITCH_ALERT_TIME = 2.5
ROLL_ABS_ON = 20.0
ROLL_ALERT_TIME = 1.5
EAR_THRESHOLD = 0.21
EYE_CLOSED_THRESHOLD = 0.18
MAR_THRESHOLD = 0.6
YAWN_LIMIT = 3

hist_pitch = deque(maxlen=SMOOTH_N)
hist_roll = deque(maxlen=SMOOTH_N)
pitch0 = None
calib_pitch = []
pitch_frames = 0
roll_frames = 0
pitch_alert_on = False
roll_alert_on = False
pitch_hold_frames = 0
blink_count = 0
yawn_count = 0
eye_closed_frames = 0
yawn_active = False

LEFT_EYE = [159, 145, 33, 133, 158, 153]
RIGHT_EYE = [386, 374, 362, 263, 385, 380]

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ====== OPEN ESP STREAM ======
ESP_URL = "http://172.20.10.3/stream"
reader = StreamReader(ESP_URL)
if not reader.connect():
    print(f"⚠ Initial connect failed — will retry in loop")

# Get FPS from first successful frame or default
fps = 30.0
calib_frames_needed = int(2.0 * fps)

# ====== MAIN LOOP ======
reconnect_delay = 2.0
last_ok = time.time()

while True:
    frame = reader.read()
    if frame is None:
        if time.time() - last_ok > reconnect_delay:
            print("⟳ Reconnecting to stream...")
            if reader.connect():
                last_ok = time.time()
                print("✓ Stream reconnected")
            else:
                time.sleep(reconnect_delay)
        else:
            time.sleep(0.1)
        continue

    last_ok = time.time()
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)
    alert_now = False

    if res.multi_face_landmarks:
        lm = res.multi_face_landmarks[0].landmark

        # Head pose
        image_points = np.array([
            (lm[IDX_NOSE_TIP].x * w, lm[IDX_NOSE_TIP].y * h),
            (lm[IDX_CHIN].x * w, lm[IDX_CHIN].y * h),
            (lm[IDX_LEFT_EYE_OUTER].x * w, lm[IDX_LEFT_EYE_OUTER].y * h),
            (lm[IDX_RIGHT_EYE_OUTER].x * w, lm[IDX_RIGHT_EYE_OUTER].y * h),
            (lm[IDX_LEFT_MOUTH].x * w, lm[IDX_LEFT_MOUTH].y * h),
            (lm[IDX_RIGHT_MOUTH].x * w, lm[IDX_RIGHT_MOUTH].y * h),
        ], dtype=np.float32)

        focal_length = w
        center = (w / 2, h / 2)
        camera_matrix = np.array([
            [focal_length, 0, center[0]],
            [0, focal_length, center[1]],
            [0, 0, 1]
        ], dtype=np.float32)
        dist_coeffs = np.zeros((4, 1))
        ok, rvec, _ = cv2.solvePnP(MODEL_POINTS, image_points, camera_matrix, dist_coeffs)
        if ok:
            R, _ = cv2.Rodrigues(rvec)
            pitch, yaw, roll = rotationMatrixToEulerAngles(R)
            hist_pitch.append(pitch)
            hist_roll.append(roll)
            pitch_s = float(np.mean(hist_pitch))
            roll_s = float(np.mean(hist_roll))

            if pitch0 is None:
                calib_pitch.append(pitch_s)
                if len(calib_pitch) >= calib_frames_needed:
                    pitch0 = float(np.mean(calib_pitch))
                    calib_pitch.clear()
                cv2.putText(frame, f"Calibrating... ({max(0, calib_frames_needed-len(calib_pitch))})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            else:
                pitch_delta = pitch_s - pitch0
                is_down_on = (pitch_delta <= -PITCH_DELTA_ON)
                is_down_off = (pitch_delta >= -PITCH_DELTA_OFF)

                if is_down_on:
                    pitch_frames += 1
                else:
                    if not pitch_alert_on:
                        pitch_frames = 0
                pitch_time = pitch_frames / fps
                if (not pitch_alert_on) and (pitch_time >= PITCH_ALERT_TIME):
                    pitch_alert_on = True
                    pitch_hold_frames = int(0.7 * fps)
                if pitch_alert_on and pitch_hold_frames > 0:
                    pitch_hold_frames -= 1
                if pitch_alert_on and pitch_hold_frames <= 0 and is_down_off:
                    pitch_alert_on = False
                    pitch_frames = 0
                if abs(roll_s) >= ROLL_ABS_ON:
                    roll_frames += 1
                else:
                    roll_frames = 0
                    roll_alert_on = False
                if (roll_frames / fps) >= ROLL_ALERT_TIME:
                    roll_alert_on = True
                if pitch_alert_on or roll_alert_on:
                    alert_now = True
                    cv2.putText(frame, "HEAD ALERT", (int(w * 0.1), int(h * 0.9)), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

            cv2.putText(frame, f"Pitch: {pitch_s:.1f}  Roll: {roll_s:.1f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Blink and yawn
        left_ear = calculate_ear(lm, LEFT_EYE)
        right_ear = calculate_ear(lm, RIGHT_EYE)
        ear = (left_ear + right_ear) / 2.0
        mar = calculate_mar(lm)

        if ear < EAR_THRESHOLD:
            blink_count += 1
        if ear < EYE_CLOSED_THRESHOLD:
            eye_closed_frames += 1
        else:
            eye_closed_frames = 0
        if mar > MAR_THRESHOLD and not yawn_active:
            yawn_count += 1
            yawn_active = True
        if mar < MAR_THRESHOLD:
            yawn_active = False

        cv2.putText(frame, f'EAR: {ear:.2f}  MAR: {mar:.2f}', (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f'Blinks: {blink_count}  Yawns: {yawn_count}', (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        if eye_closed_frames > 20:
            cv2.putText(frame, "EYES CLOSED!", (250, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        if yawn_count >= YAWN_LIMIT:
            cv2.putText(frame, "DROWSY DETECTED!", (200, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    else:
        cv2.putText(frame, "No face detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Drowsiness + Head Pose", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

reader.close()
cv2.destroyAllWindows()

Error: Could not open video stream
0.0


: 

In [17]:

import cv2
import time
import math
import numpy as np
from collections import deque
import mediapipe as mp

def euclidean_distance(p1, p2):
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

def calculate_ear(landmarks, eye_indices):
    top = landmarks[eye_indices[0]]
    bottom = landmarks[eye_indices[1]]
    left = landmarks[eye_indices[2]]
    right = landmarks[eye_indices[3]]
    v1 = landmarks[eye_indices[4]]
    v2 = landmarks[eye_indices[5]]

    vertical_1 = euclidean_distance(top, bottom)
    vertical_2 = euclidean_distance(v1, v2)
    horizontal = euclidean_distance(left, right)
    return (vertical_1 + vertical_2) / (2.0 * horizontal) if horizontal != 0 else 0.0

def calculate_mar(landmarks):
    upper = landmarks[13]
    lower = landmarks[14]
    left = landmarks[78]
    right = landmarks[308]
    vertical = euclidean_distance(upper, lower)
    horizontal = euclidean_distance(left, right)
    return vertical / horizontal if horizontal != 0 else 0.0

IDX_NOSE_TIP = 1
IDX_CHIN = 152
IDX_LEFT_EYE_OUTER = 33
IDX_RIGHT_EYE_OUTER = 263
IDX_LEFT_MOUTH = 61
IDX_RIGHT_MOUTH = 291

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),
    (0.0, -63.6, -12.5),
    (-43.3, 32.7, -26.0),
    (43.3, 32.7, -26.0),
    (-28.9, -28.9, -24.1),
    (28.9, -28.9, -24.1)
], dtype=np.float32)

def wrap180(a):
    return float((a + 180.0) % 360.0 - 180.0)

def rotationMatrixToEulerAngles(R):
    sy = math.sqrt(R[0,0]**2 + R[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.atan2(R[2,1], R[2,2])
        yaw = math.atan2(-R[2,0], sy)
        roll = math.atan2(R[1,0], R[0,0])
    else:
        pitch = math.atan2(-R[1,2], R[1,1])
        yaw = math.atan2(-R[2,0], sy)
        roll = 0.0
    pitch, yaw, roll = np.degrees([pitch, yaw, roll])
    return wrap180(pitch), wrap180(yaw), wrap180(roll)

# =========================
# HEAD POSE PARAMETERS
# =========================
SMOOTH_N = 15
PITCH_DELTA_ON = 10.0
PITCH_DELTA_OFF = 7.0
PITCH_ALERT_TIME = 2.0   # (2 sec alert)
PITCH_DANGER_TIME = 4.0   #(4 sec danger)
ROLL_ABS_ON = 20.0
ROLL_ALERT_TIME = 1.5

hist_pitch = deque(maxlen=SMOOTH_N)
hist_roll = deque(maxlen=SMOOTH_N)
pitch0 = None
calib_pitch = []
pitch_frames = 0
roll_frames = 0
pitch_alert_on = False
pitch_danger_on = False
roll_alert_on = False
pitch_hold_frames = 0

# =========================
# EYE + YAWN
# =========================
EAR_THRESHOLD = 0.21
EYE_CLOSED_THRESHOLD = 0.18
MAR_THRESHOLD = 0.6
YAWN_LIMIT = 3

blink_count = 0
yawn_count = 0
yawn_active = False

eye_close_start = None
eye_alert = False
eye_danger = False

LEFT_EYE = [159, 145, 33, 133, 158, 153]
RIGHT_EYE = [386, 374, 362, 263, 385, 380]

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)
#cap = cv2.VideoCapture("http://10.204.189.15:81/stream")

if not cap.isOpened():
    print("Error: Could not open video stream")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

calib_frames_needed = int(2.0 * fps)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)

    if res.multi_face_landmarks:
        lm = res.multi_face_landmarks[0].landmark

        # =========================
        # HEAD POSE (UPDATED LOGIC)
        # =========================
        image_points = np.array([
            (lm[IDX_NOSE_TIP].x * w, lm[IDX_NOSE_TIP].y * h),
            (lm[IDX_CHIN].x * w, lm[IDX_CHIN].y * h),
            (lm[IDX_LEFT_EYE_OUTER].x * w, lm[IDX_LEFT_EYE_OUTER].y * h),
            (lm[IDX_RIGHT_EYE_OUTER].x * w, lm[IDX_RIGHT_EYE_OUTER].y * h),
            (lm[IDX_LEFT_MOUTH].x * w, lm[IDX_LEFT_MOUTH].y * h),
            (lm[IDX_RIGHT_MOUTH].x * w, lm[IDX_RIGHT_MOUTH].y * h),
        ], dtype=np.float32)

        focal_length = w
        center = (w / 2, h / 2)

        camera_matrix = np.array([
            [focal_length, 0, center[0]],
            [0, focal_length, center[1]],
            [0, 0, 1]
        ], dtype=np.float32)

        dist_coeffs = np.zeros((4, 1))

        ok, rvec, _ = cv2.solvePnP(MODEL_POINTS, image_points, camera_matrix, dist_coeffs)

        if ok:
            R, _ = cv2.Rodrigues(rvec)
            pitch, yaw, roll = rotationMatrixToEulerAngles(R)

            hist_pitch.append(pitch)
            hist_roll.append(roll)

            pitch_s = float(np.mean(hist_pitch))
            roll_s = float(np.mean(hist_roll))

            if pitch0 is None:
                calib_pitch.append(pitch_s)
                if len(calib_pitch) >= calib_frames_needed:
                    pitch0 = float(np.mean(calib_pitch))
                    calib_pitch.clear()
            else:
                pitch_delta = pitch_s - pitch0

                is_down_on = (pitch_delta <= -PITCH_DELTA_ON)
                is_down_off = (pitch_delta >= -PITCH_DELTA_OFF)

                if is_down_on:
                    pitch_frames += 1
                else:
                    if not pitch_alert_on:
                        pitch_frames = 0

                pitch_time = pitch_frames / fps

                # ALERT (2 sec)
                if (not pitch_alert_on) and pitch_time >= PITCH_ALERT_TIME:
                    pitch_alert_on = True
                    pitch_hold_frames = int(0.7 * fps)

                # DANGER (4 sec)
                if pitch_time >= PITCH_DANGER_TIME:
                    pitch_danger_on = True
                else:
                    pitch_danger_on = False

                if pitch_alert_on and pitch_hold_frames > 0:
                    pitch_hold_frames -= 1

                if pitch_alert_on and pitch_hold_frames <= 0 and is_down_off:
                    pitch_alert_on = False
                    pitch_frames = 0
                    pitch_danger_on = False

                if abs(roll_s) >= ROLL_ABS_ON:
                    roll_frames += 1
                else:
                    roll_frames = 0
                    roll_alert_on = False

                if (roll_frames / fps) >= ROLL_ALERT_TIME:
                    roll_alert_on = True

                if pitch_danger_on:
                    cv2.putText(frame, "HEAD DANGER", (int(w * 0.1), int(h * 0.9)),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

                elif pitch_alert_on or roll_alert_on:
                    cv2.putText(frame, "HEAD ALERT", (int(w * 0.1), int(h * 0.9)),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

            cv2.putText(frame, f"Pitch: {pitch_s:.1f}  Roll: {roll_s:.1f}",
                        (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # =========================
        # EYE TIMER
        # =========================
        left_ear = calculate_ear(lm, LEFT_EYE)
        right_ear = calculate_ear(lm, RIGHT_EYE)
        ear = (left_ear + right_ear) / 2.0

        if ear < EYE_CLOSED_THRESHOLD:
            if eye_close_start is None:
                eye_close_start = time.time()

            elapsed = time.time() - eye_close_start

            if elapsed >= 2.0:
                eye_alert = True
            if elapsed >= 4.0:
                eye_danger = True
        else:
            eye_close_start = None
            eye_alert = False
            eye_danger = False

        if eye_danger:
            cv2.putText(frame, "EYE DANGER", (250, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        elif eye_alert:
            cv2.putText(frame, "EYE ALERT", (250, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)

        # =========================
        # YAWN
        # =========================
        mar = calculate_mar(lm)

        if mar > MAR_THRESHOLD and not yawn_active:
            yawn_count += 1
            yawn_active = True

        if mar < MAR_THRESHOLD:
            yawn_active = False

        if yawn_count >= 7:
            cv2.putText(frame, "YAWN DANGER", (200, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        elif yawn_count >= 5:
            cv2.putText(frame, "DROWSY (YAWN)", (200, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)

        cv2.putText(frame, f'EAR: {ear:.2f}', (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.putText(frame, f'Yawns: {yawn_count}', (10, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    else:
        cv2.putText(frame, "No face detected", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Drowsiness + Head Pose", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2

url = "http://172.20.10.3/stream"

cap = cv2.VideoCapture(url)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Reconnecting...")
        cap.release()
        cap = cv2.VideoCapture(url)
        continue

    cv2.imshow("ESP32-CAM", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#model + mediapipe

In [1]:
import cv2
import numpy as np
from collections import deque
import mediapipe as mp
import tensorflow as tf
import time
import math

# =========================
# Config - Eyes Detection
# =========================
MODEL_PATH = r"C:\Users\rwank\OneDrive\Desktop\SAV\model\eyes_int8.tflite"
IMG = 128

CLOSE_ON  = 0.67
CLOSE_OFF = 0.55

SMOOTH_N = 11
MARGIN = 0.25
CAM_W, CAM_H = 640, 480

# =========================
# Load TFLite INT8 model
# =========================
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]

def predict_open_prob(eye_bgr: np.ndarray) -> float:
    eye_rgb = cv2.cvtColor(eye_bgr, cv2.COLOR_BGR2RGB)
    eye_rgb = cv2.resize(eye_rgb, (IMG, IMG), interpolation=cv2.INTER_AREA)
    x = eye_rgb.astype(np.float32)

    xq = (x / in_scale + in_zero).astype(np.int8)[None, ...]
    interpreter.set_tensor(inp["index"], xq)
    interpreter.invoke()

    yq = interpreter.get_tensor(out["index"])[0][0]
    y = (yq - out_zero) * out_scale
    return float(y)

# =========================
# MediaPipe FaceMesh setup
# =========================
mp_face_mesh = mp.solutions.face_mesh

LEFT_EYE_IDS = set()
RIGHT_EYE_IDS = set()
for a, b in mp_face_mesh.FACEMESH_LEFT_EYE:
    LEFT_EYE_IDS.add(a); LEFT_EYE_IDS.add(b)
for a, b in mp_face_mesh.FACEMESH_RIGHT_EYE:
    RIGHT_EYE_IDS.add(a); RIGHT_EYE_IDS.add(b)

EYE_IDS = LEFT_EYE_IDS.union(RIGHT_EYE_IDS)

def eye_bbox_from_landmarks(lms, w: int, h: int):
    xs, ys = [], []
    for idx in EYE_IDS:
        x = int(lms[idx].x * w)
        y = int(lms[idx].y * h)
        xs.append(x); ys.append(y)

    if not xs:
        return None

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    bw = x2 - x1
    bh = y2 - y1

    x1 = int(x1 - MARGIN * bw); x2 = int(x2 + MARGIN * bw)
    y1 = int(y1 - MARGIN * bh); y2 = int(y2 + MARGIN * bh)

    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w - 1, x2); y2 = min(h - 1, y2)

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

# =========================
# Head Pose
# =========================
def euclidean_distance(p1, p2):
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

def calculate_mar(landmarks):
    upper = landmarks[13]
    lower = landmarks[14]
    left = landmarks[78]
    right = landmarks[308]
    vertical = euclidean_distance(upper, lower)
    horizontal = euclidean_distance(left, right)
    return vertical / horizontal if horizontal != 0 else 0.0

IDX_NOSE_TIP = 1
IDX_CHIN = 152
IDX_LEFT_EYE_OUTER = 33
IDX_RIGHT_EYE_OUTER = 263
IDX_LEFT_MOUTH = 61
IDX_RIGHT_MOUTH = 291

MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),
    (0.0, -63.6, -12.5),
    (-43.3, 32.7, -26.0),
    (43.3, 32.7, -26.0),
    (-28.9, -28.9, -24.1),
    (28.9, -28.9, -24.1)
], dtype=np.float32)

def wrap180(a):
    return float((a + 180.0) % 360.0 - 180.0)

def rotationMatrixToEulerAngles(R):
    sy = math.sqrt(R[0,0]**2 + R[1,0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.atan2(R[2,1], R[2,2])
        yaw = math.atan2(-R[2,0], sy)
        roll = math.atan2(R[1,0], R[0,0])
    else:
        pitch = math.atan2(-R[1,2], R[1,1])
        yaw = math.atan2(-R[2,0], sy)
        roll = 0.0
    pitch, yaw, roll = np.degrees([pitch, yaw, roll])
    return wrap180(pitch), wrap180(yaw), wrap180(roll)

# =========================
# HEAD TIMING
# =========================
SMOOTH_N_HEAD = 15
PITCH_DELTA_ON = 10.0
PITCH_DELTA_OFF = 7.0

PITCH_ALERT_TIME = 2.0
PITCH_DANGER_TIME = 4.0

hist_pitch = deque(maxlen=SMOOTH_N_HEAD)
hist_roll = deque(maxlen=SMOOTH_N_HEAD)

pitch0 = None
calib_pitch = []

head_down_start = None
head_alert = False
head_danger = False

# =========================
# Yawn
# =========================
MAR_THRESHOLD = 0.6
YAWN_DROWSY = 5
YAWN_DANGER = 7

yawn_count = 0
yawn_active = False

# =========================
# Eyes
# =========================
hist_close = deque(maxlen=SMOOTH_N)
state = "OPEN"

# ===== ADDED (EYE TIMING) =====
eye_close_start = None
eye_alert = False
eye_danger = False

# =========================
# Camera
# =========================
#cap = cv2.VideoCapture("http://172.20.10.3/stream")
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

calib_frames_needed = int(2.0 * fps)

with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = face_mesh.process(rgb)

        if res.multi_face_landmarks:
            lms = res.multi_face_landmarks[0].landmark
            lm = lms

            box = eye_bbox_from_landmarks(lms, w, h)

            if box is not None:
                x1, y1, x2, y2 = box
                eye_crop = frame[y1:y2, x1:x2]

                p_open = predict_open_prob(eye_crop)
                p_open = 1.0 - p_open
                p_close = 1.0 - p_open

                hist_close.append(p_close)
                avg_close = float(sum(hist_close) / len(hist_close))

                if state == "OPEN" and avg_close >= CLOSE_ON:
                    state = "CLOSED"
                elif state == "CLOSED" and avg_close <= CLOSE_OFF:
                    state = "OPEN"

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{state} close_prob(avg)={avg_close:.2f}",
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                            (255, 255, 255), 2)

                # ===== ADDED EYE TIMER LOGIC =====
                if state == "CLOSED":
                    if eye_close_start is None:
                        eye_close_start = time.time()

                    elapsed = time.time() - eye_close_start

                    if elapsed >= 2.0:
                        eye_alert = True
                    if elapsed >= 4.0:
                        eye_danger = True
                else:
                    eye_close_start = None
                    eye_alert = False
                    eye_danger = False

                if eye_danger:
                    cv2.putText(frame, "EYE DANGER", (50, 400),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                elif eye_alert:
                    cv2.putText(frame, "EYE ALERT", (50, 400),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)

            else:
                hist_close.clear()

            image_points = np.array([
                (lm[IDX_NOSE_TIP].x * w, lm[IDX_NOSE_TIP].y * h),
                (lm[IDX_CHIN].x * w, lm[IDX_CHIN].y * h),
                (lm[IDX_LEFT_EYE_OUTER].x * w, lm[IDX_LEFT_EYE_OUTER].y * h),
                (lm[IDX_RIGHT_EYE_OUTER].x * w, lm[IDX_RIGHT_EYE_OUTER].y * h),
                (lm[IDX_LEFT_MOUTH].x * w, lm[IDX_LEFT_MOUTH].y * h),
                (lm[IDX_RIGHT_MOUTH].x * w, lm[IDX_RIGHT_MOUTH].y * h),
            ], dtype=np.float32)

            focal_length = w
            center = (w / 2, h / 2)

            camera_matrix = np.array([
                [focal_length, 0, center[0]],
                [0, focal_length, center[1]],
                [0, 0, 1]
            ], dtype=np.float32)

            dist_coeffs = np.zeros((4, 1))

            ok, rvec, _ = cv2.solvePnP(MODEL_POINTS, image_points, camera_matrix, dist_coeffs)

            if ok:
                R, _ = cv2.Rodrigues(rvec)
                pitch, yaw, roll = rotationMatrixToEulerAngles(R)

                hist_pitch.append(pitch)
                pitch_s = float(np.mean(hist_pitch))

                if pitch0 is None:
                    calib_pitch.append(pitch_s)
                    if len(calib_pitch) >= calib_frames_needed:
                        pitch0 = float(np.mean(calib_pitch))
                        calib_pitch.clear()
                else:
                    pitch_delta = pitch_s - pitch0

                    if pitch_delta <= -PITCH_DELTA_ON:
                        if head_down_start is None:
                            head_down_start = time.time()

                        elapsed = time.time() - head_down_start

                        if elapsed >= PITCH_ALERT_TIME:
                            head_alert = True

                        if elapsed >= PITCH_DANGER_TIME:
                            head_danger = True
                    else:
                        head_down_start = None
                        head_alert = False
                        head_danger = False

                    if head_danger:
                        cv2.putText(frame, "HEAD DANGER", (50, 450),
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                    elif head_alert:
                        cv2.putText(frame, "HEAD ALERT", (50, 450),
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)

            mar = calculate_mar(lm)

            if mar > MAR_THRESHOLD and not yawn_active:
                yawn_count += 1
                yawn_active = True

            if mar < MAR_THRESHOLD:
                yawn_active = False

            if yawn_count >= YAWN_DANGER:
                cv2.putText(frame, "DANGER!", (200, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            elif yawn_count >= YAWN_DROWSY:
                cv2.putText(frame, "DROWSY", (200, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)

            cv2.putText(frame, f"Yawn Count: {yawn_count}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        cv2.imshow("ALL SYSTEM", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()